# External Framework Validation — SDMetrics

**Clean, linear pipeline.** Run top to bottom (Kernel -> Restart & Run All).

This notebook evaluates the CTGAN and CTAB-GAN+ synthetic datasets with
**SDMetrics** (DataCebo / SDV, originally MIT Data to AI Lab). SDMetrics is
model-agnostic and organises its evaluation around two reports plus a large set
of individually applicable metrics.

Following the agreed full scope, this notebook computes:

* the **QualityReport** (overall quality score, Column Shapes, Column Pair
  Trends),
* the **DiagnosticReport** (data validity and data structure),
* a set of **individual metrics** grouped by role: shape (KSComplement,
  TVComplement), boundary/validity (BoundaryAdherence, CategoryAdherence,
  RangeCoverage, CategoryCoverage), novelty (NewRowSynthesis), and SDMetrics' own privacy
  metrics (DCRBaselineProtection, DCROverfittingProtection) which connect
  directly to the manual DCR analysis.

Metrics that duplicate SynthEval / Synthcity (KS, correlation, detection) are
retained here for cross-framework triangulation, not as new findings. The
genuinely additive parts are the single overall quality score, the diagnostic
validity checks (which relate to the CTGAN domain violations found by
the domain-feasibility audit), NewRowSynthesis, and the SDMetrics DCR-based privacy metrics.

To stay consistent with the other frameworks, the same files are used:
`real_minority_train.csv` as the real reference and
`synthetic_ctgan_final.csv` / `synthetic_ctab_final.csv` as the synthetic sets.

**Requirements:** environment `tese_sdmetrics` (Python 3.10) with
`sdmetrics==0.28.2`.


## 0. Configuration and paths

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

RANDOM_STATE = 123

# >>> Adjust this path if necessary <<<
DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)

import sdmetrics
print("sdmetrics version:", sdmetrics.__version__)


Data folder: data
sdmetrics version: 0.28.2


## 1. Canonical column definitions

Identical to the other framework notebooks. `CAT_COLS` (SEX, EDUCATION,
MARRIAGE, PAY_1..PAY_6) plus the target are treated as categorical; the rest are
numerical. This mirrors SynthEval Variant A and the Synthcity discrete-column
declaration, so that all frameworks share the same categorical/numerical split.


In [2]:
TARGET   = "def_pay"

FEATURES = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
            "PAY_1", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
            "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
            "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

ALL_COLS = FEATURES + [TARGET]

MONETARY = ["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1, 7)] \
                         + [f"PAY_AMT{i}"  for i in range(1, 7)]

CAT_COLS = ["SEX", "EDUCATION", "MARRIAGE"] + [f"PAY_{i}" for i in range(1, 7)]
CAT_WITH_TARGET = CAT_COLS + [TARGET]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

assert len(FEATURES) == 23 and len(CAT_COLS) == 9 and len(NUM_COLS) == 14
print("FEATURES:", len(FEATURES), "| CAT:", len(CAT_COLS), "| NUM:", len(NUM_COLS))


FEATURES: 23 | CAT: 9 | NUM: 14


## 2. Build the SDMetrics metadata dictionary

SDMetrics needs a metadata dictionary describing each column's `sdtype`
(`numerical` or `categorical`). The categorical columns and the target are
declared categorical, the monetary and remaining numeric columns as numerical.
This is the SDMetrics analogue of `cat_cols` in the other frameworks.


In [3]:
metadata = {"columns": {}}
for col in ALL_COLS:
    if col in CAT_WITH_TARGET:
        metadata["columns"][col] = {"sdtype": "categorical"}
    else:
        metadata["columns"][col] = {"sdtype": "numerical"}

n_cat = sum(1 for c in metadata["columns"].values() if c["sdtype"] == "categorical")
n_num = sum(1 for c in metadata["columns"].values() if c["sdtype"] == "numerical")
print(f"metadata: {n_cat} categorical, {n_num} numerical, {len(metadata['columns'])} total")


metadata: 10 categorical, 14 numerical, 24 total


## 3. Helper: align a dataframe to the canonical schema

Keeps the 23 features plus target, enforces order, casts monetary columns to
`float64`, and casts categorical columns to a consistent type so that SDMetrics
does not misread integer-coded categories. All three datasets (real and the two
synthetic) must share the same columns.


In [4]:
def align(df, name):
    d = df.copy()
    assert TARGET in d.columns, f"{name}: '{TARGET}' column is required"
    d = d[ALL_COLS]
    d[MONETARY] = d[MONETARY].astype("float64")
    # categorical (incl. target) as string labels for stable SDMetrics typing
    for c in CAT_WITH_TARGET:
        d[c] = d[c].astype(int).astype(str)
    assert list(d.columns) == ALL_COLS
    print(f"{name:<16} -> {d.shape}")
    return d


## 4. Load the real and synthetic datasets

In [5]:
real      = align(pd.read_csv(DATA_DIR / "real_minority_train.csv"),  "real (min)")
syn_ctgan = align(pd.read_csv(DATA_DIR / "synthetic_ctgan_final.csv"), "syn ctgan")
syn_ctab  = align(pd.read_csv(DATA_DIR / "synthetic_ctab_final.csv"),  "syn ctabgan+")

for d in (syn_ctgan, syn_ctab):
    assert list(d.columns) == list(real.columns)
print("\nColumns aligned.")


real (min)       -> (4645, 24)
syn ctgan        -> (11710, 24)
syn ctabgan+     -> (11710, 24)

Columns aligned.


## 5. Reconstruct the canonical split → holdout

Rebuilds the 70/30 split and verifies it against `real_train_full.csv`, as in
the other notebooks. The minority holdout is used as the "control" real data for
the SDMetrics DCR-based privacy metrics, which compare distances to the training
data against distances to a held-out set.


In [6]:
df_raw = pd.read_csv(DATA_DIR / "UCI_Credit_Card.csv")
df_c = (df_raw.drop(columns=["ID"])
              .rename(columns={"PAY_0": "PAY_1",
                               "default.payment.next.month": TARGET}))
df_c["EDUCATION"] = df_c["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df_c["MARRIAGE"]  = df_c["MARRIAGE"].replace({0: 3})

X = df_c.drop(columns=TARGET); y = df_c[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

real_full = pd.read_csv(DATA_DIR / "real_train_full.csv")
train_recon = X_train.copy(); train_recon[TARGET] = y_train
train_recon = train_recon[real_full.columns].reset_index(drop=True)
assert train_recon.equals(real_full.reset_index(drop=True)), \
    "Reconstructed split != real_train_full.csv"
print("Split reproduced bit for bit. OK.")

holdout = X_test.copy(); holdout[TARGET] = y_test
holdout_min = align(holdout[holdout[TARGET].astype(int) == 1], "holdout (min)")
print("Minority holdout:", holdout_min.shape)


Split reproduced bit for bit. OK.
holdout (min)    -> (1991, 24)
Minority holdout: (1991, 24)


## 6. Quality Report

Captures the statistical similarity between real and synthetic data as a single
overall score, decomposed into Column Shapes (per-column distributions) and
Column Pair Trends (pairwise relationships).


In [7]:
from sdmetrics.reports.single_table import QualityReport

def quality(syn, name):
    rep = QualityReport()
    rep.generate(real, syn, metadata)
    props = rep.get_properties()
    overall = rep.get_score()
    print(f"{name}: overall = {overall:.4f}")
    print(props.to_string(index=False))
    row = {"model": name, "overall_quality": overall}
    for _, r in props.iterrows():
        row[r["Property"]] = r["Score"]
    return row, rep

q_ctgan, rep_q_ctgan = quality(syn_ctgan, "CTGAN")
print()
q_ctab, rep_q_ctab = quality(syn_ctab, "CTAB-GAN+")

quality_rows = [q_ctgan, q_ctab]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 24/24 [00:00<00:00, 287.36it/s]|
Column Shapes Score: 84.13%

(2/2) Evaluating Column Pair Trends: |██████████| 276/276 [00:00<00:00, 310.91it/s]|
Column Pair Trends Score: 72.87%

Overall Score (Average): 78.5%

CTGAN: overall = 0.7850
          Property    Score
     Column Shapes 0.841269
Column Pair Trends 0.728697

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 24/24 [00:00<00:00, 340.40it/s]|
Column Shapes Score: 82.62%

(2/2) Evaluating Column Pair Trends: |██████████| 276/276 [00:00<00:00, 327.13it/s]|
Column Pair Trends Score: 88.23%

Overall Score (Average): 85.43%

CTAB-GAN+: overall = 0.8543
          Property   Score
     Column Shapes 0.82621
Column Pair Trends 0.88229


## 7. Diagnostic Report

A quick validity check: are the synthetic values valid (within bounds, correct
categories) and is the structure correct. Low validity here would correspond to
the domain violations found by the domain-feasibility audit.


In [8]:
from sdmetrics.reports.single_table import DiagnosticReport

def diagnostic(syn, name):
    rep = DiagnosticReport()
    rep.generate(real, syn, metadata)
    props = rep.get_properties()
    overall = rep.get_score()
    print(f"{name}: overall diagnostic = {overall:.4f}")
    print(props.to_string(index=False))
    row = {"model": name, "overall_diagnostic": overall}
    for _, r in props.iterrows():
        row[r["Property"]] = r["Score"]
    return row

d_ctgan = diagnostic(syn_ctgan, "CTGAN")
print()
d_ctab = diagnostic(syn_ctab, "CTAB-GAN+")

diagnostic_rows = [d_ctgan, d_ctab]


Generating report ...

(1/2) Evaluating Data Validity: |██████████| 24/24 [00:00<00:00, 999.53it/s]|
Data Validity Score: 99.93%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 400.30it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 99.96%

CTGAN: overall diagnostic = 0.9996
      Property    Score
 Data Validity 0.999253
Data Structure 1.000000

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 24/24 [00:00<00:00, 1021.27it/s]|
Data Validity Score: 99.94%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 333.54it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 99.97%

CTAB-GAN+: overall diagnostic = 0.9997
      Property    Score
 Data Validity 0.999427
Data Structure 1.000000


## 8. Individual column-shape metrics (triangulation)

KSComplement for numerical columns and TVComplement for categorical columns,
averaged over their respective column groups. These are complements (1 =
identical distributions), so they triangulate with SynthEval KS/TVD (which are
distances) once the inversion is accounted for.


In [9]:
from sdmetrics.single_column import (
    KSComplement, TVComplement, BoundaryAdherence,
    CategoryAdherence, RangeCoverage, CategoryCoverage,
)

def col_shapes(syn, name):
    ks  = np.mean([KSComplement.compute(real[c], syn[c]) for c in NUM_COLS])
    tv  = np.mean([TVComplement.compute(real[c], syn[c]) for c in CAT_WITH_TARGET])
    ba  = np.mean([BoundaryAdherence.compute(real[c], syn[c]) for c in NUM_COLS])
    ca  = np.mean([CategoryAdherence.compute(real[c], syn[c]) for c in CAT_WITH_TARGET])
    rc  = np.mean([RangeCoverage.compute(real[c], syn[c]) for c in NUM_COLS])
    cc  = np.mean([CategoryCoverage.compute(real[c], syn[c]) for c in CAT_WITH_TARGET])
    row = {"model": name, "KSComplement_num": ks, "TVComplement_cat": tv,
           "BoundaryAdherence": ba, "CategoryAdherence": ca,
           "RangeCoverage": rc, "CategoryCoverage": cc}
    print(f"{name}: KS={ks:.4f} TV={tv:.4f} Boundary={ba:.4f} "
          f"CatAdh={ca:.4f} RangeCov={rc:.4f} CatCov={cc:.4f}")
    return row

shape_rows = [col_shapes(syn_ctgan, "CTGAN"), col_shapes(syn_ctab, "CTAB-GAN+")]


CTGAN: KS=0.8325 TV=0.8536 Boundary=0.9987 CatAdh=1.0000 RangeCov=0.7321 CatCov=1.0000
CTAB-GAN+: KS=0.7739 TV=0.8995 Boundary=0.9991 CatAdh=0.9999 RangeCov=0.7705 CatCov=0.9489


## 9. Novelty: NewRowSynthesis

Measures the fraction of synthetic rows that are genuinely new rather than exact
copies of real rows. A value near 1 means little direct copying (good for
privacy); a low value indicates memorisation. `numerical_match_tolerance`
controls how close two numeric values must be to count as a match.


In [12]:
from sdmetrics.single_table import NewRowSynthesis

def novelty(syn, name):
    score = NewRowSynthesis.compute(
        real_data=real, synthetic_data=syn, metadata=metadata,
        numerical_match_tolerance=0.01,
        synthetic_sample_size=min(5000, len(syn)),
    )
    print(f"{name}: NewRowSynthesis = {score:.4f}")
    return {"model": name, "NewRowSynthesis": score}

novelty_rows = [novelty(syn_ctgan, "CTGAN"), novelty(syn_ctab, "CTAB-GAN+")]


CTGAN: NewRowSynthesis = 1.0000
CTAB-GAN+: NewRowSynthesis = 1.0000


## 10. SDMetrics privacy metrics (DCR-based)

SDMetrics provides its own distance-to-closest-record privacy metrics.
`DCRBaselineProtection` compares the synthetic-to-real distances against a
random-data baseline. `DCROverfittingProtection` checks whether the synthetic
data is closer to the training data than to a held-out set (a sign of
overfitting or leakage). These connect directly to the manual DCR/NNDR analysis
and offer an independent implementation of the same idea.

Note: the minority holdout (1,991 records) is smaller than half of the
training reference (4,645), below the size SDMetrics recommends for
`DCROverfittingProtection`. The library's warning is silenced by
`warnings.filterwarnings("ignore")`; the score is therefore less precise.


In [13]:
from sdmetrics.single_table import DCRBaselineProtection, DCROverfittingProtection

def dcr_privacy(syn, name):
    base = DCRBaselineProtection.compute(
        real_data=real, synthetic_data=syn, metadata=metadata)
    over = DCROverfittingProtection.compute(
        real_training_data=real, synthetic_data=syn,
        real_validation_data=holdout_min, metadata=metadata)
    # both return dicts in recent versions; keep the headline score
    base_s = base["score"] if isinstance(base, dict) else base
    over_s = over["score"] if isinstance(over, dict) else over
    print(f"{name}: DCRBaseline={base_s:.4f}  DCROverfitting={over_s:.4f}")
    return {"model": name, "DCRBaselineProtection": base_s,
            "DCROverfittingProtection": over_s}

privacy_rows = [dcr_privacy(syn_ctgan, "CTGAN"), dcr_privacy(syn_ctab, "CTAB-GAN+")]


CTGAN: DCRBaseline=0.1670  DCROverfitting=0.4822
CTAB-GAN+: DCRBaseline=0.0613  DCROverfitting=0.4804


## 11. Consolidate and save results

All blocks are merged per model and written to a single CSV, following the
naming convention of the other frameworks.


In [14]:
from functools import reduce

frames = []
for rows in (quality_rows, diagnostic_rows, shape_rows, novelty_rows, privacy_rows):
    frames.append(pd.DataFrame(rows).set_index("model"))

sdmetrics_all = reduce(lambda a, b: a.join(b), frames).reset_index()
sdmetrics_all.to_csv(DATA_DIR / "sdmetrics_all_results.csv", index=False)
print("Saved: sdmetrics_all_results.csv\n")

with pd.option_context("display.max_columns", None, "display.width", 200):
    print(sdmetrics_all.round(4).to_string(index=False))


Saved: sdmetrics_all_results.csv

    model  overall_quality  Column Shapes  Column Pair Trends  overall_diagnostic  Data Validity  Data Structure  KSComplement_num  TVComplement_cat  BoundaryAdherence  CategoryAdherence  RangeCoverage  CategoryCoverage  NewRowSynthesis  DCRBaselineProtection  DCROverfittingProtection
    CTGAN           0.7850         0.8413              0.7287              0.9996         0.9993             1.0            0.8325            0.8536             0.9987             1.0000         0.7321            1.0000              1.0                 0.1670                    0.4822
CTAB-GAN+           0.8543         0.8262              0.8823              0.9997         0.9994             1.0            0.7739            0.8995             0.9991             0.9999         0.7705            0.9489              1.0                 0.0613                    0.4804


---
**Done.** If every cell ran, `sdmetrics_all_results.csv` holds the consolidated
scores and `sdmetrics_all` is available in memory. Note the direction of each
metric when interpreting: the report scores, complements, adherence, coverage
and NewRowSynthesis are all "higher is better", while the two DCR protection
metrics are also "higher is better" (more protection).
